In [1]:
import pandas as pd
import faiss
import torch
import json
from datetime import datetime
from diskcache import Cache
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import numpy as np

In [2]:
# -----------------------------
# Load necessary models & data
# -----------------------------

# Load chunked text data
df = pd.read_csv(r"C:\Users\hp\Downloads\filtered_chunks.csv")
texts = df["chunk_text"].tolist()

# Load FAISS index
index = faiss.read_index(r"C:/Users/hp/Downloads/faiss_index1a.idx")

# Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")


In [3]:
# Load TinyLLaMA model + tokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
llama_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
).to("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# -----------------------------
# Utility Functions
# -----------------------------

def retrieve_top_k_chunks(query, k=5):
    query_vector = embed_model.encode([query]).astype("float32")
    D, I = index.search(query_vector, k)
    return [texts[i] for i in I[0]]


In [5]:
def build_prompt(query, chunks):
    context = "\n\n".join(chunks)
    return f"""You are an intelligent assistant. Use the following context to answer the question clearly and briefly. If unsure,say "I don't know".

Context:
{context}

Question: {query}
"""

In [6]:
def fallback_prompt(query, chunks):
    context = "\n".join(chunks[:1])  # TinyLLaMA handles small context better
    return f"""Use the following context to answer the question briefly.

Context: {context}

Question: {query}

Answer:"""

In [7]:
def call_tiny_llama(prompt):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(llama_model.device)
    with torch.no_grad():
        output = llama_model.generate(input_ids, max_new_tokens=200)
    return tokenizer.decode(output[0], skip_special_tokens=True)


In [8]:
def call_openai(prompt):
    raise Exception("Simulated OpenAI failure for fallback testing.")

In [9]:
def clean_answer(raw_output):
    if "Answer:" in raw_output:
        return raw_output.split("Answer:")[-1].strip().split("\n")[0]
    return raw_output.strip().split("\n")[0]


In [10]:
def is_gibberish(text):
    return len(text.strip()) < 20 or any(bad in text for bad in ["....", "andand", "s.s.s", ",.,.,", "0000"])


In [12]:
#----------logging function------------
def log_query(query, chunks, prompt, answer, model_used, user_feedback=None, file_path="query_logs.json"):
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "retrieved_chunks": chunks,
        "prompt": prompt,
        "answer": answer,
        "model_used": model_used,
        "user_feedback": user_feedback
    }
    with open(file_path, "a", encoding="utf-8") as f:
        json.dump(log_entry, f)
        f.write("\n")

In [22]:
# -----------------------------
# RAG Pipeline
# -----------------------------

def rag_pipeline(query, feedback=None):
    chunks = retrieve_top_k_chunks(query)
    prompt = build_prompt(query, chunks)

    try:
        raw_answer = call_openai(prompt)
        answer = clean_answer(raw_answer)
        model_used = "OpenAI"
    except:
        print("OpenAI failed. Using TinyLLaMA fallback.")
        fallback = fallback_prompt(query, chunks)
        raw_answer = call_tiny_llama(fallback)
        answer = clean_answer(raw_answer)
        model_used = "TinyLLaMA"

    # Fallback if the answer is still gibberish
    if is_gibberish(answer):
        answer = "Sorry, I couldn't generate a clear answer from the documents."

    log_query(query, chunks, prompt, answer, model_used, user_feedback=feedback)
    return answer

In [23]:
query = "Who is marieve herington?"
prompt = f"Answer the question clearly and step by step:\n\n{query}"
print(call_tiny_llama(prompt))

Answer the question clearly and step by step:

Who is marieve herington?

Answer: Marieve Herington is a British actress who has appeared in numerous films and TV shows, including "The Crown," "The Durrells," and "The Night Manager."


In [24]:
#for clearing cache
from diskcache import Cache

# Load the cache
cache = Cache("./llm_cache")

# Clear all entries in the cache
cache.clear()

print("Cache completely cleared.")


Cache completely cleared.


In [25]:
# -----------------------------
# Caching Logic + Execution
# -----------------------------

if __name__ == "__main__":
    cache = Cache("./llm_cache")
    query = "Who is Marieve Herington?"
    feedback = None

    if query in cache:
        print("Retrieved from cache.")
        final_answer = cache[query]
    else:
        final_answer = rag_pipeline(query, feedback)
        cache[query] = final_answer

    print("\n Final Answer:\n", final_answer)

OpenAI failed. Using TinyLLaMA fallback.

 Final Answer:
 Marieve Herington is an American singer, songwriter, and actress. She is best known for her work in the animated series Sesame Park, where she sang the theme songs for CBC Television's Sesame Park and Nelvana's Pippi Longstocking and Marigold S Mathemagics. Herington also appeared on the TV series Marigold S Mathemagics and Marigold S Mathemagics: The Musical. She has also performed in Los Angeles and Toronto.


In [26]:
if __name__ == "__main__":
    cache = Cache("./llm_cache")
    query = "Who provides the voice of the animated lead characters in the TV series Delilah julius and Pearlie?"
    feedback = None

    if query in cache:
        print("Retrieved from cache.")
        final_answer = cache[query]
    else:
        final_answer = rag_pipeline(query, feedback)
        cache[query] = final_answer

    print("\n Final Answer:\n", final_answer) 

OpenAI failed. Using TinyLLaMA fallback.

 Final Answer:
 The voice of Delilah julius is provided by actress and singer jessica chastain. The voice of Pearlie is provided by actress and singer kristen bell.


In [27]:
if __name__ == "__main__":
    cache = Cache("./llm_cache")
    queries = [
        "Which team has the most points in the NHL, based on the provided data?",
        "How do you describe Honoka's character as a force of nature?"
    ]
    feedback = None

    for query in queries:
        if query in cache:
            print(f"Retrieved from cache for query: {query}")
            final_answer = cache[query]
        else:
            final_answer = rag_pipeline(query, feedback)
            cache[query] = final_answer

        print(f"\nFinal Answer for query:\n{query}\n{final_answer}\n")


OpenAI failed. Using TinyLLaMA fallback.

Final Answer for query:
Which team has the most points in the NHL, based on the provided data?
The Pittsburgh Penguins have the most points in the NHL, with a total of 101 points.

OpenAI failed. Using TinyLLaMA fallback.

Final Answer for query:
How do you describe Honoka's character as a force of nature?
Honoka is a force of nature, a character that is both powerful and unpredictable. She is described as a "wildfire" that "burns with a fierce intensity" and "drives the winds with a force that can't be contained." Honoka's power is both a blessing and a curse, as she can be both a force for good and a force for destruction. She is often described as being "unpredictable" and "uncontrollable," which suggests that she is not always under the control of her own actions. However, Honoka's unpredictability also makes her a force to be reckoned with, as she can cause destruction and chaos with a single touch. Overall, Honoka is a character that is b